# 🦜⛓️ LangChain 101 — Guide Complet

> **Objectif** : Maîtriser LangChain de zéro à la construction d'un système RAG complet.

---

## 📋 Plan du notebook

| # | Section | Description |
|---|---------|-------------|
| 1 | **Qu'est-ce que LangChain ?** | Vue d'ensemble, histoire, positionnement |
| 2 | **Pourquoi utiliser LangChain ?** | Avantages, cas d'usage, comparatif |
| 3 | **Architecture & Concepts de base** | Composants principaux |
| 4 | **LLMs & Chat Models** | Initialisation, invocation |
| 5 | **Prompt Templates** | PromptTemplate, ChatPromptTemplate, FewShot |
| 6 | **Chains** | LLMChain, SequentialChain, LCEL |
| 7 | **Memory** | Types de mémoire, gestion du contexte |
| 8 | **Document Loaders** | Chargement de différents types de fichiers |
| 9 | **Text Splitters** | Stratégies de chunking |
| 10 | **Embeddings & Vectorstores** | Représentation vectorielle, FAISS, Chroma |
| 11 | **Retrievers** | Recherche sémantique, MMR, self-query |
| 12 | **Construire un RAG complet** | Pipeline end-to-end |
| 13 | **Agents & Tools** | Autonomie avec les LLMs |
| 14 | **Bonnes pratiques** | Production, sécurité, optimisation |

---

## 0. 📦 Installation des dépendances

In [ ]:
# Installation des packages nécessaires
%pip install -q langchain langchain-openai langchain-community \
    langchain-chroma chromadb faiss-cpu \
    pypdf tiktoken python-dotenv openai

In [ ]:
import os
from dotenv import load_dotenv

# Charger les variables d'environnement depuis un fichier .env
load_dotenv()

# ⚠️  Remplacez par votre clé API ou utilisez un fichier .env
# os.environ["OPENAI_API_KEY"] = "sk-..."

print("✅ Environnement configuré")
print(f"Clé API présente : {'OPENAI_API_KEY' in os.environ}")

---

## 1. 🤔 Qu'est-ce que LangChain ?

### Définition

**LangChain** est un framework open-source (Python & JavaScript) conçu pour **simplifier le développement d'applications basées sur des Large Language Models (LLMs)**.

Lancé en **octobre 2022** par Harrison Chase, il est rapidement devenu le framework de référence de l'écosystème LLM.

### Historique

| Date | Événement |
|------|-----------|
| Oct 2022 | Première version open-source |
| Jan 2023 | 10 000 ⭐ GitHub en 1 semaine |
| Apr 2023 | Levée de fonds 10M$ (seed) |
| Sep 2023 | LangChain v0.1 stable |
| Jan 2024 | LangChain v0.1 + LCEL (LangChain Expression Language) |
| 2024+ | LangGraph, LangServe, LangSmith |

### Vue d'ensemble de l'écosystème LangChain

```
┌──────────────────────────────────────────────────────────────┐
│                    ÉCOSYSTÈME LANGCHAIN                      │
├──────────────────┬────────────────────┬─────────────────────┤
│   LangChain Core │    LangSmith       │    LangGraph        │
│   (Framework)    │   (Observability)  │  (Workflows/Agents) │
├──────────────────┴────────────────────┴─────────────────────┤
│                    LangServe (Déploiement)                   │
├──────────────────────────────────────────────────────────────┤
│              LangChain Hub (Templates & Prompts)             │
└──────────────────────────────────────────────────────────────┘
```

### Structure des packages Python

| Package | Rôle |
|---------|------|
| `langchain-core` | Abstractions de base, LCEL |
| `langchain` | Chains, agents, stratégies de retrieval |
| `langchain-community` | Intégrations tierces (FAISS, Chroma, Ollama…) |
| `langchain-openai` | Intégration OpenAI/Azure |
| `langchain-anthropic` | Intégration Anthropic (Claude) |
| `langchain-google-genai` | Intégration Google Gemini |
| `langgraph` | Workflows d'agents avec état |

---

## 2. 💡 Pourquoi utiliser LangChain ?

### Problèmes que LangChain résout

| Problème sans LangChain | Solution LangChain |
|------------------------|-------------------|
| Code boilerplate pour chaque appel API | Abstractions unifiées |
| Pas de mémoire entre les requêtes | Module Memory intégré |
| Difficile de chaîner plusieurs étapes | Chains et LCEL |
| Ingestion de documents complexe | Document Loaders prêts à l'emploi |
| Recherche sémantique à implémenter | Vectorstores + Retrievers intégrés |
| Prompt engineering fastidieux | PromptTemplates réutilisables |
| Agents à coder from scratch | Framework d'agents clé en main |
| Debugging opaque | LangSmith observability |

### Cas d'usage principaux

```
📄 RAG (Question-Réponse sur documents)
    └── Chat sur PDF, base de connaissances interne

🤖 Chatbots & Assistants conversationnels
    └── Mémoire, personnalité, contexte

🔍 Extraction d'information structurée
    └── Parsing, classification, NER

🛠️ Agents autonomes
    └── Utilisation d'outils, navigation web, code

📊 Analyse de données
    └── Résumé, synthèse, rapports automatiques

🔄 Automatisation de workflows
    └── Pipelines intelligents, traitement en batch
```

### LangChain vs alternatives

| Framework | Points forts | Points faibles |
|-----------|-------------|---------------|
| **LangChain** | Riche, flexible, grande communauté | Peut être verbeux |
| **LlamaIndex** | Excellent pour le RAG | Moins généraliste |
| **Haystack** | Production-ready | Courbe d'apprentissage |
| **Semantic Kernel** | Intégration Microsoft | Orienté C#/.NET |
| **LangGraph** | Workflows complexes | Complémentaire à LangChain |
| **From scratch** | Contrôle total | Long à développer |

---

## 3. 🏗️ Architecture & Concepts de base

### Les 6 composants fondamentaux

```
┌─────────────────────────────────────────────────────────────┐
│                    APPLICATION LANGCHAIN                     │
│                                                             │
│  ┌──────────┐  ┌──────────┐  ┌──────────┐  ┌──────────┐  │
│  │   LLMs   │  │ Prompts  │  │  Chains  │  │  Memory  │  │
│  │          │  │Templates │  │  (LCEL)  │  │          │  │
│  └──────────┘  └──────────┘  └──────────┘  └──────────┘  │
│                                                             │
│  ┌──────────────────────────┐  ┌─────────────────────────┐ │
│  │    Document Pipeline     │  │       Agents            │ │
│  │ Loaders→Splitters→Embed  │  │   Tools + ReAct loop    │ │
│  │ →Vectorstore→Retriever   │  │                         │ │
│  └──────────────────────────┘  └─────────────────────────┘ │
└─────────────────────────────────────────────────────────────┘
```

| Composant | Rôle | Exemple |
|-----------|------|---------|
| **LLMs / Chat Models** | Interface avec le modèle de langage | `ChatOpenAI`, `ChatAnthropic` |
| **Prompt Templates** | Formatage dynamique des prompts | `PromptTemplate`, `ChatPromptTemplate` |
| **Chains** | Orchestration d'étapes | `LLMChain`, pipelines LCEL |
| **Memory** | Persistance du contexte | `ConversationBufferMemory` |
| **Document Pipeline** | Ingestion et recherche | Loaders → Splitters → Vectorstore |
| **Agents** | Actions autonomes avec outils | `ReActAgent`, `OpenAIFunctionsAgent` |

---

## 4. 🤖 LLMs & Chat Models

### Différence entre LLM et ChatModel

| Aspect | LLM (texte → texte) | ChatModel (messages → message) |
|--------|---------------------|--------------------------------|
| **Input** | String brute | Liste de messages |
| **Output** | String brute | AIMessage |
| **Usage** | Complétion | Conversation |
| **Exemple** | `OpenAI(model="gpt-3.5-turbo-instruct")` | `ChatOpenAI(model="gpt-4o")` |
| **Recommandé** | Cas simples | ✅ Cas courants (2024+) |

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# ─────────────────────────────────────
# Initialiser un Chat Model
# ─────────────────────────────────────
llm = ChatOpenAI(
    model="gpt-3.5-turbo",   # Modèle à utiliser
    temperature=0.7,          # Créativité : 0.0 (déterministe) → 2.0 (créatif)
    max_tokens=500,           # Limite de tokens en sortie
)

print("✅ ChatOpenAI initialisé")
print(f"  Modèle : {llm.model_name}")
print(f"  Température : {llm.temperature}")

In [ ]:
# ─────────────────────────────────────
# Invocation simple
# ─────────────────────────────────────
response = llm.invoke("Explique le machine learning en 2 phrases.")

print("Type de réponse :", type(response).__name__)
print("Contenu         :", response.content)
print("Métadonnées     :", response.response_metadata)

In [ ]:
# ─────────────────────────────────────
# Invocation avec messages structurés
# ─────────────────────────────────────
messages = [
    SystemMessage(content="Tu es un professeur expert en data science. Réponds en français."),
    HumanMessage(content="Quelle est la différence entre deep learning et machine learning ?")
]

response = llm.invoke(messages)
print(response.content)

In [ ]:
# ─────────────────────────────────────
# Streaming
# ─────────────────────────────────────
print("Réponse en streaming :")
print("-" * 40)

for chunk in llm.stream("Cite 5 langages de programmation populaires en data science."):
    print(chunk.content, end="", flush=True)

print("\n" + "-" * 40)

### Modèles disponibles

| Fournisseur | Modèles | Package |
|-------------|---------|--------|
| **OpenAI** | gpt-4o, gpt-4-turbo, gpt-3.5-turbo | `langchain-openai` |
| **Anthropic** | claude-3-5-sonnet, claude-3-opus | `langchain-anthropic` |
| **Google** | gemini-1.5-pro, gemini-1.5-flash | `langchain-google-genai` |
| **Mistral** | mistral-large, mistral-small | `langchain-mistralai` |
| **Ollama** | llama3, mistral, phi3 (local) | `langchain-community` |
| **Groq** | llama3-70b, mixtral (ultra-rapide) | `langchain-groq` |

---

## 5. 📝 Prompt Templates

### Pourquoi les templates ?

Les **Prompt Templates** permettent de :
- 🔄 **Réutiliser** les prompts avec différentes entrées
- 🧹 **Séparer** la logique du contenu
- ✅ **Valider** les variables d'entrée
- 🧱 **Composer** des prompts complexes

In [ ]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.prompts import FewShotPromptTemplate

# ─────────────────────────────────────
# 1. PromptTemplate basique
# ─────────────────────────────────────
template = PromptTemplate(
    input_variables=["sujet", "niveau"],
    template="""Tu es un expert pédagogue.
Explique '{sujet}' à un étudiant de niveau {niveau}.
Utilise des analogies simples et des exemples concrets."""
)

# Formater avec des variables
prompt_formate = template.format(sujet="les réseaux de neurones", niveau="débutant")
print("Prompt formaté :")
print("-" * 40)
print(prompt_formate)

In [ ]:
# ─────────────────────────────────────
# 2. ChatPromptTemplate (recommandé)
# ─────────────────────────────────────
chat_template = ChatPromptTemplate.from_messages([
    ("system", "Tu es un assistant spécialisé en {domaine}. Réponds toujours en {langue}."),
    ("human", "Question : {question}")
])

# Formater en messages
messages = chat_template.format_messages(
    domaine="data science",
    langue="français",
    question="Comment fonctionne le gradient descent ?"
)

print("Messages générés :")
for msg in messages:
    print(f"  [{msg.__class__.__name__}] {msg.content[:60]}...")

In [ ]:
# ─────────────────────────────────────
# 3. Few-Shot Prompting
# ─────────────────────────────────────
exemples = [
    {"texte": "J'adore ce produit, il est fantastique!", "sentiment": "positif"},
    {"texte": "Vraiment décevant, ne fonctionne pas.",  "sentiment": "négatif"},
    {"texte": "Le colis est arrivé aujourd'hui.",        "sentiment": "neutre"},
]

exemple_template = PromptTemplate(
    input_variables=["texte", "sentiment"],
    template="Texte: {texte}\nSentiment: {sentiment}"
)

few_shot_prompt = FewShotPromptTemplate(
    examples=exemples,
    example_prompt=exemple_template,
    prefix="Classifie le sentiment des textes suivants:",
    suffix="Texte: {input}\nSentiment:",
    input_variables=["input"]
)

print(few_shot_prompt.format(input="Ce film était vraiment bien réalisé!"))

### Récapitulatif des Prompt Templates

| Template | Usage | Avantage |
|----------|-------|----------|
| `PromptTemplate` | Prompt texte simple | Simple, flexible |
| `ChatPromptTemplate` | Conversations multi-tours | ✅ Recommandé pour ChatModels |
| `FewShotPromptTemplate` | Exemples dans le prompt | Améliore la précision |
| `FewShotChatMessagePromptTemplate` | Few-shot + chat | Combine les deux |
| `PipelinePromptTemplate` | Composition de templates | Réutilisabilité maximale |

---

## 6. ⛓️ Chains & LCEL

### LCEL — LangChain Expression Language

La **LCEL** est l'approche moderne de LangChain pour composer des pipelines.
Elle utilise l'opérateur `|` (pipe) pour chaîner les composants.

```
prompt | llm | output_parser
   ↓        ↓         ↓
formate  génère   extrait
```

**Avantages LCEL :**
- ✅ Streaming natif
- ✅ Traitement par batch
- ✅ Parallélisation avec `RunnableParallel`
- ✅ Async natif
- ✅ Fallbacks et retries

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# ─────────────────────────────────────
# Chain simple avec LCEL : prompt | llm | parser
# ─────────────────────────────────────
prompt = ChatPromptTemplate.from_template(
    "Explique {sujet} en exactement 3 points clés. Sois concis."
)

# Le pipe | crée une Runnable chain
chain = prompt | llm | StrOutputParser()

# Invocation
resultat = chain.invoke({"sujet": "le RAG en intelligence artificielle"})
print(resultat)

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

# ─────────────────────────────────────
# Chain séquentielle (étape 1 → étape 2)
# ─────────────────────────────────────

# Étape 1 : résumer
resume_prompt = ChatPromptTemplate.from_template(
    "Résume ce texte en 2 phrases : {texte}"
)
chain_resume = resume_prompt | llm | StrOutputParser()

# Étape 2 : traduire le résumé
traduction_prompt = ChatPromptTemplate.from_template(
    "Traduis ce texte en anglais : {resume}"
)
chain_traduction = traduction_prompt | llm | StrOutputParser()

# Pipeline complet : texte → résumé → traduction
pipeline = (
    {"resume": chain_resume}  # Passe le résumé à l'étape suivante
    | chain_traduction
)

texte_test = """
LangChain est un framework révolutionnaire qui simplifie le développement 
d'applications basées sur les grands modèles de langage. Il offre des 
abstractions puissantes pour les chains, la mémoire et les agents.
"""

# Exécuter en batch
print("Résumé :")
resume = chain_resume.invoke({"texte": texte_test})
print(resume)
print("\nTraduction :")
print(chain_traduction.invoke({"resume": resume}))

In [ ]:
# ─────────────────────────────────────
# RunnableParallel : exécution en parallèle
# ─────────────────────────────────────

prompt_resume = ChatPromptTemplate.from_template("Résume en 1 phrase : {texte}")
prompt_themes = ChatPromptTemplate.from_template("Liste 3 thèmes principaux : {texte}")
prompt_ton = ChatPromptTemplate.from_template(
    "Décris le ton (formel/informel/neutre) : {texte}"
)

analyse_parallele = RunnableParallel(
    resume  = prompt_resume  | llm | StrOutputParser(),
    themes  = prompt_themes  | llm | StrOutputParser(),
    ton     = prompt_ton     | llm | StrOutputParser(),
)

resultat = analyse_parallele.invoke({"texte": texte_test})

print("📊 Analyse parallèle :")
for cle, valeur in resultat.items():
    print(f"\n  [{cle.upper()}]")
    print(f"  {valeur}")

### Récapitulatif des Chains

| Type | Usage | LCEL Équivalent |
|------|-------|----------------|
| Simple chain | prompt → llm → output | `prompt \| llm \| parser` |
| Sequential | étape1 → étape2 | `chain1 \| chain2` |
| Parallel | 2 tâches en même temps | `RunnableParallel(a=..., b=...)` |
| Conditional | branchement selon condition | `RunnableBranch(...)` |
| With fallback | retry si erreur | `chain.with_fallbacks([...])` |

---

## 7. 🧠 Memory — Gestion du contexte conversationnel

La **mémoire** permet au LLM de se souvenir des échanges précédents dans une conversation.

### Types de mémoire

| Type | Stockage | Tokens utilisés | Idéal pour |
|------|----------|-----------------|------------|
| `ConversationBufferMemory` | Tout l'historique | 📈 Croissant | Conversations courtes |
| `ConversationBufferWindowMemory` | N derniers messages | 🔒 Fixe | Chatbots production |
| `ConversationSummaryMemory` | Résumé progressif | 📊 Modéré | Longues conversations |
| `ConversationSummaryBufferMemory` | Résumé + récents | 📊 Optimal | ✅ Recommandé |
| `ConversationTokenBufferMemory` | Limite en tokens | 🔒 Fixe | Contrôle des coûts |
| `VectorStoreRetrieverMemory` | Recherche sémantique | 🔍 Pertinent | Très longues sessions |

In [ ]:
from langchain.memory import ConversationBufferWindowMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# ─────────────────────────────────────
# Mémoire avec LCEL (approche moderne)
# ─────────────────────────────────────

# Prompt avec placeholder pour l'historique
prompt_conv = ChatPromptTemplate.from_messages([
    ("system", "Tu es un assistant amical. Réponds en français."),
    MessagesPlaceholder(variable_name="historique"),  # ← Injecte l'historique ici
    ("human", "{input}")
])

chain_conv = prompt_conv | llm | StrOutputParser()

# Gestion des sessions de conversation
store = {}  # Stocke les historiques par session_id

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# Wrapper avec mémoire
conversation = RunnableWithMessageHistory(
    chain_conv,
    get_session_history,
    input_messages_key="input",
    history_messages_key="historique"
)

config = {"configurable": {"session_id": "demo_session"}}

# Tour 1
r1 = conversation.invoke({"input": "Je m'appelle Alice et j'apprends Python."}, config=config)
print("Tour 1 :", r1)

# Tour 2 - le modèle se souvient du nom
r2 = conversation.invoke({"input": "Quel est mon prénom ?"}, config=config)
print("\nTour 2 :", r2)

---

## 8. 📂 Document Loaders

Les **Document Loaders** chargent des données depuis diverses sources et les transforment en objets `Document`.

### Structure d'un Document
```python
Document(
    page_content="texte du document...",
    metadata={"source": "fichier.pdf", "page": 1, ...}
)
```

### Loaders disponibles

| Source | Loader | Package |
|--------|--------|--------|
| **PDF** | `PyPDFLoader` | `pypdf` |
| **Word** | `Docx2txtLoader` | `docx2txt` |
| **CSV** | `CSVLoader` | intégré |
| **TXT** | `TextLoader` | intégré |
| **HTML** | `BSHTMLLoader` | `beautifulsoup4` |
| **Notion** | `NotionDBLoader` | API Notion |
| **Wikipedia** | `WikipediaLoader` | `wikipedia` |
| **URL** | `WebBaseLoader` | `requests` |
| **YouTube** | `YoutubeLoader` | `youtube-transcript-api` |
| **JSON** | `JSONLoader` | `jq` |
| **Directory** | `DirectoryLoader` | intégré |

In [ ]:
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain.schema import Document
import tempfile

# ─────────────────────────────────────
# Exemple avec un fichier texte
# ─────────────────────────────────────

# Créer un fichier texte temporaire pour la démo
with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False, encoding='utf-8') as f:
    f.write("""Le Machine Learning est une branche de l'IA qui permet aux machines d'apprendre.

Le Deep Learning utilise des réseaux de neurones profonds.
Il excelle dans la reconnaissance d'images et le NLP.

LangChain est un framework pour construire des applications LLM.
Il simplifie l'orchestration des modèles de langage.
""")
    tmp_path = f.name

# Charger le document
loader = TextLoader(tmp_path, encoding='utf-8')
documents = loader.load()

print(f"Nombre de documents chargés : {len(documents)}")
print(f"Type               : {type(documents[0]).__name__}")
print(f"Contenu (100 chars): {documents[0].page_content[:100]}")
print(f"Métadonnées        : {documents[0].metadata}")

In [ ]:
# ─────────────────────────────────────
# Créer des Documents manuellement
# ─────────────────────────────────────
docs_manuels = [
    Document(
        page_content="LangChain facilite le développement d'applications LLM.",
        metadata={"source": "intro.txt", "auteur": "Harrison Chase", "annee": 2022}
    ),
    Document(
        page_content="Le RAG combine recherche vectorielle et génération de texte.",
        metadata={"source": "rag_guide.txt", "categorie": "technique"}
    ),
]

print(f"Documents créés : {len(docs_manuels)}")
for i, doc in enumerate(docs_manuels, 1):
    print(f"  Doc {i}: '{doc.page_content[:50]}...' | meta={doc.metadata}")

---

## 9. ✂️ Text Splitters

Le **Text Splitting** (ou chunking) découpe les documents en morceaux plus petits pour :
- Respecter les limites de contexte du modèle
- Améliorer la précision de la recherche
- Réduire les coûts

### Stratégies de chunking

| Splitter | Séparateur | Idéal pour |
|----------|------------|------------|
| `RecursiveCharacterTextSplitter` | `\n\n` → `\n` → ` ` | ✅ Usage général |
| `CharacterTextSplitter` | Caractère unique | Texte simple |
| `TokenTextSplitter` | Tokens (tiktoken) | Contrôle précis des tokens |
| `MarkdownHeaderTextSplitter` | Titres Markdown | Documentation |
| `HTMLHeaderTextSplitter` | Balises HTML | Pages web |
| `SentenceTransformersTokenTextSplitter` | Tokens BERT | NLP avancé |
| `SemanticChunker` | Coupures sémantiques | ✅ Meilleure qualité |

### Paramètres clés

| Paramètre | Description | Recommandation |
|-----------|-------------|----------------|
| `chunk_size` | Taille max du chunk | 500–1000 tokens |
| `chunk_overlap` | Chevauchement entre chunks | 10–20% du chunk_size |
| `length_function` | Fonction de mesure de longueur | `len` ou `tiktoken` |

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ─────────────────────────────────────
# RecursiveCharacterTextSplitter
# ─────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,       # Taille maximale d'un chunk (en caractères)
    chunk_overlap=30,     # Chevauchement pour conserver le contexte
    length_function=len,  # Fonction de mesure (len ou tiktoken)
    add_start_index=True, # Ajoute la position de départ dans metadata
)

texte_long = """
LangChain est un framework open-source créé par Harrison Chase en 2022.
Il simplifie le développement d'applications basées sur les LLMs.

Les composants principaux sont les Chains, la Memory, les Agents et le RAG.
Chaque composant peut être utilisé indépendamment ou combiné avec d'autres.

Le RAG (Retrieval-Augmented Generation) est l'un des cas d'usage les plus populaires.
Il combine la recherche vectorielle avec la génération de texte pour répondre aux questions.
"""

chunks = splitter.split_text(texte_long)

print(f"Texte original   : {len(texte_long)} caractères")
print(f"Nombre de chunks : {len(chunks)}")
print()
for i, chunk in enumerate(chunks, 1):
    print(f"Chunk {i} ({len(chunk)} chars): '{chunk[:60].strip()}...'") 

In [ ]:
# ─────────────────────────────────────
# Splitter sur des Documents
# ─────────────────────────────────────
chunks_docs = splitter.split_documents(docs_manuels)

print(f"Documents originaux : {len(docs_manuels)}")
print(f"Chunks générés      : {len(chunks_docs)}")
print()
for chunk in chunks_docs:
    print(f"  Source: {chunk.metadata.get('source', 'N/A')} | '{chunk.page_content[:60]}...'")

---

## 10. 🔢 Embeddings & Vectorstores

### Qu'est-ce qu'un Embedding ?

Un **embedding** est une représentation vectorielle d'un texte dans un espace multidimensionnel.
Des textes sémantiquement similaires ont des vecteurs proches.

```
"chat"       → [0.12, -0.45, 0.78, ...]  (1536 dimensions pour OpenAI)
"félin"      → [0.11, -0.43, 0.76, ...]  ← proche de "chat"
"ordinateur" → [-0.82, 0.33, -0.21, ...] ← loin de "chat"
```

### Modèles d'embedding

| Modèle | Dimensions | Coût | Performance |
|--------|-----------|------|-------------|
| `text-embedding-3-large` (OpenAI) | 3072 | 💰 Payant | ⭐⭐⭐⭐⭐ |
| `text-embedding-3-small` (OpenAI) | 1536 | 💰 Faible | ⭐⭐⭐⭐ |
| `text-embedding-ada-002` (OpenAI) | 1536 | 💰 Faible | ⭐⭐⭐⭐ |
| `all-MiniLM-L6-v2` (HuggingFace) | 384 | 🆓 Gratuit | ⭐⭐⭐ |
| `nomic-embed-text` (Ollama) | 768 | 🆓 Local | ⭐⭐⭐ |

### Vectorstores populaires

| Vectorstore | Type | Points forts |
|-------------|------|-------------|
| **FAISS** | Local | ✅ Rapide, gratuit, léger |
| **Chroma** | Local / Cloud | ✅ Facile à utiliser |
| **Pinecone** | Cloud managé | Scalable, production |
| **Weaviate** | Open-source | Filtres avancés |
| **Qdrant** | Open-source | Haute performance |
| **PGVector** | PostgreSQL | Si PostgreSQL déjà utilisé |

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# ─────────────────────────────────────
# Initialiser les embeddings OpenAI
# ─────────────────────────────────────
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Tester sur un texte
vecteur = embeddings.embed_query("LangChain pour le RAG")
print(f"Dimensions du vecteur : {len(vecteur)}")
print(f"Premiers éléments     : {vecteur[:5]}")

In [ ]:
# ─────────────────────────────────────
# Créer un Vectorstore FAISS
# ─────────────────────────────────────
textes = [
    "LangChain est un framework pour les LLMs",
    "Le RAG combine retrieval et génération",
    "Python est populaire en data science",
    "Les transformers révolutionnent le NLP",
    "FAISS est une bibliothèque de recherche vectorielle de Meta",
    "Les embeddings représentent le texte en vecteurs",
]

# Créer le vectorstore à partir des textes
vectorstore = FAISS.from_texts(textes, embeddings)

print("✅ Vectorstore FAISS créé")
print(f"Nombre de documents indexés : {vectorstore.index.ntotal}")

In [ ]:
# ─────────────────────────────────────
# Recherche de similarité
# ─────────────────────────────────────
query = "Comment fonctionne la recherche sémantique ?"

# Recherche top-3
resultats = vectorstore.similarity_search(query, k=3)

print(f"Query : '{query}'")
print(f"\nTop {len(resultats)} résultats :")
for i, doc in enumerate(resultats, 1):
    print(f"  {i}. {doc.page_content}")

In [ ]:
# ─────────────────────────────────────
# Recherche avec scores
# ─────────────────────────────────────
resultats_avec_scores = vectorstore.similarity_search_with_score(query, k=3)

print(f"Query : '{query}'")
print()
for doc, score in resultats_avec_scores:
    print(f"  Score (L2) : {score:.4f} | Texte : {doc.page_content}")
    
print("\n⚠️  Pour FAISS, un score L2 plus PETIT = plus SIMILAIRE")

---

## 11. 🔍 Retrievers

Un **Retriever** est une interface standardisée pour récupérer des documents pertinents.
Il abstrait le vectorstore et expose une méthode `get_relevant_documents(query)`.

### Types de Retrievers

| Retriever | Description | Quand l'utiliser |
|-----------|-------------|------------------|
| **Similarity Search** | Recherche par similarité cosinus | Cas général |
| **MMR** (Max Marginal Relevance) | Diversité + pertinence | Éviter la redondance |
| **Self-Query** | LLM génère la query + filtres | Filtrage par métadonnées |
| **Contextual Compression** | Compresse les chunks récupérés | Réduire le bruit |
| **Multi-Query** | Génère plusieurs queries | Améliorer le rappel |
| **BM25** | Recherche lexicale (TF-IDF) | Textes courts, mots-clés |
| **Ensemble** | Combine plusieurs retrievers | ✅ Production |

In [ ]:
# ─────────────────────────────────────
# Retriever simple (Similarity Search)
# ─────────────────────────────────────
retriever = vectorstore.as_retriever(
    search_type="similarity",  # ou "mmr"
    search_kwargs={"k": 3}     # Nombre de documents à récupérer
)

docs_recuperes = retriever.invoke("Qu'est-ce que LangChain ?")
print(f"Documents récupérés : {len(docs_recuperes)}")
for doc in docs_recuperes:
    print(f"  → {doc.page_content}")

In [ ]:
# ─────────────────────────────────────
# MMR Retriever (Max Marginal Relevance)
# ─────────────────────────────────────
retriever_mmr = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 10,   # Récupère 10, retient les 3 plus diversifiés
        "lambda_mult": 0.5  # 0 = diversité max, 1 = pertinence max
    }
)

docs_mmr = retriever_mmr.invoke("framework IA")
print("Résultats MMR (diversifiés) :")
for doc in docs_mmr:
    print(f"  → {doc.page_content}")

---

## 12. 🔧 Construire un RAG Complet

### Architecture d'un RAG

```
                    ┌─────────────────────────────────────┐
                    │        PHASE D'INDEXATION           │
                    └─────────────────────────────────────┘

  Documents  ──▶  Loader  ──▶  Splitter  ──▶  Embeddings  ──▶  Vectorstore
  (PDF, TXT)                  (chunks)        (vecteurs)       (FAISS)


                    ┌─────────────────────────────────────┐
                    │         PHASE DE REQUÊTE            │
                    └─────────────────────────────────────┘

  Question  ──▶  Embeddings  ──▶  Retriever  ──▶  Contexte
  (user)         (query vec)       (top-k)           │
                                                     ▼
  Réponse  ◀──   LLM   ◀──  Prompt Template  ◀──  {question + contexte}
```

### Les deux types de RAG

| Type | Description | Avantages |
|------|-------------|----------|
| **Naïf (Basic RAG)** | Query → Retrieve → Generate | Simple, rapide |
| **Advanced RAG** | Rewrite + Retrieve + Rerank + Generate | Plus précis |
| **Modular RAG** | Composants interchangeables | Flexible |

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# ─────────────────────────────────────
# ÉTAPE 1 : Préparer les documents
# ─────────────────────────────────────
docs_rag = [
    Document(
        page_content="""LangChain est un framework open-source créé par Harrison Chase en octobre 2022.
        Il permet de construire des applications basées sur les LLMs.
        Ses composants principaux sont : Chains, Memory, Agents, et Document Pipelines.""",
        metadata={"source": "intro_langchain.txt"}
    ),
    Document(
        page_content="""Le RAG (Retrieval-Augmented Generation) est une technique qui enrichit les prompts
        avec des documents pertinents récupérés depuis une base vectorielle.
        Il permet au LLM d'accéder à des connaissances externes sans réentraînement.""",
        metadata={"source": "rag_guide.txt"}
    ),
    Document(
        page_content="""FAISS (Facebook AI Similarity Search) est une bibliothèque pour la recherche
        de voisins les plus proches dans des espaces vectoriels de haute dimension.
        Elle est optimisée pour la vitesse et peut gérer des milliards de vecteurs.""",
        metadata={"source": "faiss_doc.txt"}
    ),
    Document(
        page_content="""Les embeddings convertissent le texte en vecteurs numériques.
        Des textes sémantiquement similaires ont des vecteurs proches.
        OpenAI propose text-embedding-3-small (1536 dimensions) et text-embedding-3-large (3072 dimensions).""",
        metadata={"source": "embeddings_guide.txt"}
    ),
]

print(f"✅ {len(docs_rag)} documents préparés")

In [ ]:
# ─────────────────────────────────────
# ÉTAPE 2 : Découper en chunks
# ─────────────────────────────────────
splitter_rag = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
)

chunks_rag = splitter_rag.split_documents(docs_rag)
print(f"✅ {len(chunks_rag)} chunks créés depuis {len(docs_rag)} documents")

In [ ]:
# ─────────────────────────────────────
# ÉTAPE 3 : Créer le vectorstore
# ─────────────────────────────────────
embeddings_rag = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore_rag = FAISS.from_documents(chunks_rag, embeddings_rag)

print(f"✅ Index FAISS créé avec {vectorstore_rag.index.ntotal} vecteurs")

In [ ]:
# ─────────────────────────────────────
# ÉTAPE 4 : Créer le retriever
# ─────────────────────────────────────
retriever_rag = vectorstore_rag.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("✅ Retriever créé")

In [ ]:
# ─────────────────────────────────────
# ÉTAPE 5 : Prompt RAG
# ─────────────────────────────────────
prompt_rag = ChatPromptTemplate.from_template("""
Tu es un assistant expert. Réponds à la question en te basant UNIQUEMENT sur le contexte fourni.
Si la réponse ne se trouve pas dans le contexte, dis clairement que tu ne sais pas.

Contexte :
{contexte}

Question : {question}

Réponse :
""")

print("✅ Prompt RAG créé")

In [ ]:
# ─────────────────────────────────────
# ÉTAPE 6 : Assembler le pipeline RAG
# ─────────────────────────────────────

def formater_docs(docs):
    """Concatène les documents récupérés en une chaîne de texte."""
    return "\n\n".join(
        f"[Source: {doc.metadata.get('source', 'Inconnu')}]\n{doc.page_content}"
        for doc in docs
    )

# Pipeline RAG avec LCEL
rag_chain = (
    {
        "contexte": retriever_rag | formater_docs,   # Récupère et formate les docs
        "question": RunnablePassthrough()            # Passe la question telle quelle
    }
    | prompt_rag
    | llm
    | StrOutputParser()
)

print("✅ Pipeline RAG assemblé")

In [ ]:
# ─────────────────────────────────────
# ÉTAPE 7 : Tester le RAG
# ─────────────────────────────────────

questions = [
    "Qu'est-ce que LangChain et qui l'a créé ?",
    "Comment fonctionne le RAG ?",
    "Quelle est la différence entre text-embedding-3-small et text-embedding-3-large ?",
    "Qu'est-ce que FAISS ?",
]

for question in questions:
    print("=" * 60)
    print(f"❓ {question}")
    print("-" * 60)
    reponse = rag_chain.invoke(question)
    print(f"💡 {reponse}")
    print()

In [ ]:
# ─────────────────────────────────────
# RAG avec sources (version enrichie)
# ─────────────────────────────────────
from langchain_core.runnables import RunnableParallel

# Garder les documents ET la réponse
rag_chain_avec_sources = RunnableParallel(
    {"docs": retriever_rag, "question": RunnablePassthrough()}
).assign(
    reponse=(
        lambda x: {
            "contexte": formater_docs(x["docs"]),
            "question": x["question"]
        }
    )
    | prompt_rag
    | llm
    | StrOutputParser()
)

resultat = rag_chain_avec_sources.invoke("Qu'est-ce que FAISS ?")

print("📝 Réponse :")
print(resultat["reponse"])
print("\n📚 Sources utilisées :")
for doc in resultat["docs"]:
    print(f"  → {doc.metadata.get('source', 'Inconnu')}")

### Récapitulatif du pipeline RAG

| Étape | Outil LangChain | Paramètres clés |
|-------|-----------------|------------------|
| **1. Charger** | `PyPDFLoader`, `TextLoader` | `file_path` |
| **2. Découper** | `RecursiveCharacterTextSplitter` | `chunk_size`, `chunk_overlap` |
| **3. Embeddings** | `OpenAIEmbeddings` | `model` |
| **4. Indexer** | `FAISS.from_documents()` | `documents`, `embedding` |
| **5. Retriever** | `vectorstore.as_retriever()` | `search_type`, `k` |
| **6. Prompt** | `ChatPromptTemplate` | Variables `{contexte}`, `{question}` |
| **7. Générer** | `chain.invoke(question)` | |

---

## 13. 🕵️ Agents & Tools

### Qu'est-ce qu'un Agent ?

Un **Agent** est un LLM qui peut :
1. **Raisonner** sur une tâche
2. **Choisir** quel outil utiliser
3. **Exécuter** l'outil
4. **Observer** le résultat
5. **Itérer** jusqu'à accomplir la tâche

```
┌─────────────────────────────────────────────┐
│                  AGENT LOOP                 │
│                                             │
│  Tâche ──▶ Réfléchir ──▶ Choisir outil     │
│                               │             │
│                          Exécuter outil     │
│                               │             │
│                         Observer résultat   │
│                               │             │
│                         Continuer / Stop    │
└─────────────────────────────────────────────┘
```

### Types d'agents

| Agent | Stratégie | Idéal pour |
|-------|-----------|------------|
| **ReAct** | Reason + Act | Usage général |
| **OpenAI Functions** | Function calling natif | ✅ GPT-4 |
| **Tool Calling** | Nouveau standard | ✅ 2024+ |
| **Self-Ask with Search** | Décompose en sous-questions | Recherche |
| **Plan and Execute** | Planifie puis exécute | Tâches complexes |

In [ ]:
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.tools import tool
import ast
import operator as op

# ─────────────────────────────────────
# Définir des outils personnalisés
# ─────────────────────────────────────

# Évaluateur arithmétique sécurisé (sans eval)
_OPS = {
    ast.Add: op.add, ast.Sub: op.sub,
    ast.Mult: op.mul, ast.Div: op.truediv,
    ast.Pow: op.pow, ast.USub: op.neg,
}

def _eval_node(node):
    """Évalue récursivement un nœud AST arithmétique."""
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    elif isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    elif isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.operand))
    else:
        raise ValueError(f"Opération non supportée : {ast.dump(node)}")

@tool
def calculatrice(expression: str) -> str:
    """Évalue une expression mathématique. Ex: '2 + 2', '10 * 5 / 2'"""
    try:
        tree = ast.parse(expression.strip(), mode='eval')
        result = _eval_node(tree.body)
        return str(round(result, 10))
    except Exception as e:
        return f"Expression non valide : {e}"

@tool
def convertisseur_temperature(valeur: float, de: str, vers: str) -> str:
    """Convertit une température entre Celsius, Fahrenheit et Kelvin."""
    de, vers = de.lower(), vers.lower()
    if de == "celsius" and vers == "fahrenheit":
        return f"{valeur}°C = {valeur * 9/5 + 32:.2f}°F"
    elif de == "fahrenheit" and vers == "celsius":
        return f"{valeur}°F = {(valeur - 32) * 5/9:.2f}°C"
    elif de == "celsius" and vers == "kelvin":
        return f"{valeur}°C = {valeur + 273.15:.2f}K"
    else:
        return f"Conversion {de} → {vers} non supportée"

outils = [calculatrice, convertisseur_temperature]

print("✅ Outils créés :")
for outil in outils:
    print(f"  • {outil.name}: {outil.description}")

In [ ]:
# ─────────────────────────────────────
# Créer l'agent
# ─────────────────────────────────────
prompt_agent = ChatPromptTemplate.from_messages([
    ("system", "Tu es un assistant utile. Utilise les outils disponibles pour répondre."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

# Créer l'agent avec tool calling
agent = create_tool_calling_agent(
    llm=ChatOpenAI(model="gpt-3.5-turbo", temperature=0),
    tools=outils,
    prompt=prompt_agent
)

# Exécuteur de l'agent
agent_executor = AgentExecutor(
    agent=agent,
    tools=outils,
    verbose=True,   # Affiche le raisonnement
    max_iterations=5
)

print("✅ Agent créé")

In [ ]:
# ─────────────────────────────────────
# Tester l'agent
# ─────────────────────────────────────
print("Test 1 : Calcul")
resultat = agent_executor.invoke({"input": "Combien font 15 * 8 + 42 / 3 ?"})
print("Réponse finale :", resultat["output"])

print("\nTest 2 : Conversion de température")
resultat2 = agent_executor.invoke({"input": "Convertis 100°C en Fahrenheit"})
print("Réponse finale :", resultat2["output"])

---

## 14. ✅ Bonnes pratiques & Production

### 1. Gestion des coûts

```python
from langchain_community.callbacks import get_openai_callback

with get_openai_callback() as cb:
    response = chain.invoke("Ma question")
    print(f"Tokens utilisés : {cb.total_tokens}")
    print(f"Coût estimé     : ${cb.total_cost:.4f}")
```

### 2. Caching

```python
from langchain.globals import set_llm_cache
from langchain.cache import InMemoryCache, SQLiteCache

# Cache en mémoire (sessions courtes)
set_llm_cache(InMemoryCache())

# Cache persistant (entre sessions)
set_llm_cache(SQLiteCache(database_path=".langchain.db"))
```

### 3. Gestion des erreurs

```python
from langchain_core.runnables import RunnableWithFallbacks

# Fallback automatique si le modèle principal échoue
chain_robuste = (
    ChatOpenAI(model="gpt-4o")
    .with_fallbacks([ChatOpenAI(model="gpt-3.5-turbo")])
)
```

### 4. Streaming asynchrone

```python
import asyncio

async def stream_reponse(question: str):
    async for chunk in chain.astream({"question": question}):
        print(chunk, end="", flush=True)

asyncio.run(stream_reponse("Explique le RAG"))
```

### 5. Sauvegarde/Chargement du vectorstore

```python
# Sauvegarder
vectorstore.save_local("mon_index_faiss")

# Charger
vectorstore_charge = FAISS.load_local(
    "mon_index_faiss", 
    embeddings,
    allow_dangerous_deserialization=True
)
```

### Récapitulatif des bonnes pratiques

| Aspect | ❌ À éviter | ✅ Recommandé |
|--------|------------|---------------|
| **Mémoire** | `ConversationBufferMemory` illimitée | `ConversationBufferWindowMemory(k=5)` |
| **Coût** | Appels répétitifs sans cache | Activer le cache LLM |
| **Chunks** | Trop petits (<100 chars) | 300–1000 tokens avec overlap |
| **Top-k** | k=1 (manque contexte) | k=3 à 5 pour le RAG |
| **Prompt** | Instructions vagues | Instructions précises avec exemples |
| **Erreurs** | Pas de try/except | Fallbacks + retry + logging |
| **Secrets** | Clé API dans le code | Variables d'environnement (.env) |
| **Monitoring** | Aucun | LangSmith en production |

---

## 🎯 Récapitulatif Général — LangChain 101

### La roadmap LangChain

```
NIVEAU 1 : Bases
├── ✅ ChatModel (ChatOpenAI)
├── ✅ PromptTemplate
└── ✅ StrOutputParser

NIVEAU 2 : Pipelines
├── ✅ LCEL (| operator)
├── ✅ RunnableParallel
└── ✅ RunnablePassthrough

NIVEAU 3 : Mémoire & Conversation
├── ✅ ChatMessageHistory
└── ✅ RunnableWithMessageHistory

NIVEAU 4 : RAG
├── ✅ Document Loaders
├── ✅ Text Splitters
├── ✅ Embeddings
├── ✅ Vectorstore (FAISS)
└── ✅ Retriever

NIVEAU 5 : Agents
├── ✅ Tools (@tool decorator)
├── ✅ create_tool_calling_agent
└── ✅ AgentExecutor

NIVEAU 6 : Production
├── ✅ Caching
├── ✅ Fallbacks
├── ✅ Monitoring (LangSmith)
└── ✅ Async / Streaming
```

### Ressources pour aller plus loin

| Ressource | URL |
|-----------|-----|
| 📚 Documentation officielle | https://python.langchain.com/docs |
| 🍳 LangChain Cookbook | https://github.com/langchain-ai/langchain/tree/master/cookbook |
| 🔍 LangSmith (monitoring) | https://smith.langchain.com |
| 🌐 LangChain Hub (prompts) | https://smith.langchain.com/hub |
| 🎥 Tutoriels vidéo | https://youtube.com/@LangChain |
| 💬 Discord | https://discord.gg/langchain |

In [ ]:
# ─────────────────────────────────────
# 🏁 Mini-projet : RAG conversationnel
# ─────────────────────────────────────
# Combining tout ce qu'on a appris !

from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Prompt RAG avec historique de conversation
prompt_rag_conv = ChatPromptTemplate.from_messages([
    ("system", """
Tu es un assistant expert. Réponds à la question en te basant sur le contexte fourni.
Si la réponse ne se trouve pas dans le contexte, dis-le clairement.

Contexte documentaire :
{contexte}
"""),
    MessagesPlaceholder(variable_name="historique"),
    ("human", "{question}")
])

# Chain RAG de base
def creer_rag_chain(retriever):
    chain_base = (
        RunnablePassthrough.assign(
            contexte=lambda x: formater_docs(retriever.invoke(x["question"]))
        )
        | prompt_rag_conv
        | llm
        | StrOutputParser()
    )
    
    sessions = {}
    def get_hist(session_id):
        if session_id not in sessions:
            sessions[session_id] = ChatMessageHistory()
        return sessions[session_id]
    
    return RunnableWithMessageHistory(
        chain_base,
        get_hist,
        input_messages_key="question",
        history_messages_key="historique"
    )

# Créer le RAG conversationnel
rag_conv = creer_rag_chain(retriever_rag)
cfg = {"configurable": {"session_id": "demo_rag"}}

print("🤖 RAG conversationnel prêt!\n")

# Simuler une conversation
q1 = "Qu'est-ce que LangChain ?"
q2 = "Et le RAG, comment ça fonctionne ?"
q3 = "Peux-tu résumer les deux concepts en une phrase chacun ?"

for q in [q1, q2, q3]:
    print(f"👤 {q}")
    rep = rag_conv.invoke({"question": q}, config=cfg)
    print(f"🤖 {rep}")
    print()

---

## 🎉 Félicitations !

Vous maîtrisez maintenant les fondamentaux de LangChain :

- ✅ **LLMs & Chat Models** — Interface unifiée avec OpenAI, Anthropic, etc.
- ✅ **Prompt Templates** — Prompts dynamiques et réutilisables
- ✅ **LCEL** — Composition de pipelines avec l'opérateur `|`
- ✅ **Memory** — Conversations avec contexte persistant
- ✅ **Document Pipeline** — Loaders → Splitters → Embeddings → Vectorstore
- ✅ **Retrievers** — Recherche sémantique avancée
- ✅ **RAG complet** — Système de Q&A sur vos documents
- ✅ **Agents** — LLMs autonomes avec outils

### Prochaines étapes

1. 🔗 Explorer **LangGraph** pour des workflows d'agents complexes
2. 📊 Utiliser **LangSmith** pour le monitoring en production
3. 🚀 Déployer avec **LangServe** (FastAPI intégré)
4. 📄 Tester l'application RAG PDF Streamlit dans le dossier `s13_langchain_rag_pdf/`

---
*Notebook créé pour la formation GrowUP AI — Session 13 LangChain*